---

## EXERCISE1: Burglary network

Implement the *Prior Sampling* algorithm to do approximate inference on last week's Burglary network.

<img src='images/burglary.png'>

Verify that the algorithm can correctly approximate the probability $P(j, m, a, \neg b, \neg e) = 0.00063$

Try different numbers of samples (e.g. $N = 10, 100, 1000, 10000$) and compare the results.

---

In [2]:
# I report the code explained during the lesson below since it is the base for solving the exercise 1
import numpy as np
import random as rnd

t, f = 0, 1

def samplegen(Pdist, Parents = []):
	assert len(Parents) < len(Pdist.shape)
	if rnd.random() < Pdist[t][tuple(Parents)]:
		return t
	return f

def parents(X):
	return [val[i] for i in par[X]]


# Now Let's define the network using the same structures presented during the lesson

# problem distributions
P_B = np.array([0.001, 0.999])
P_E = np.array([0.002, 0.998])
P_A_BE = np.array([[[0.95, 0.94],[0.29, 0.001]],
				   [[0.05, 0.06],[0.71, 0.999]]])
P_J_A = np.array([[0.9, 0.05],[0.1, 0.95]])
P_M_A = np.array([[0.7, 0.01],[0.3, 0.99]])


# network variables...
var = ['B','E','A','J','M']
# their distributions...
prd = {'B':P_B, 'E':P_E, 'A':P_A_BE, 'J':P_J_A, 'M':P_M_A}
# their parents...
par = {'B':[],'E':[], 'A':['B','E'], 'J':['A'], 'M':['A']}
# and their initial values
val = {'B':f, 'E':f, 'A':f, 'J':f, 'M':f}


In [3]:
# Given the request to apply prior sampling with different number of samples, I create a function with the code explained during the lesson and with the param
#the number of events
def priorSampling(n_events):
  event = []

  for n in range(n_events):
    for x in var:
      val[x] = samplegen(prd[x], parents(x))
    event.append(['f' if val[x] else 't' for x in var])

  print("Number or randomly generated events = ", len(event))
  return event

In [5]:
#Q.1.1 Try different numbers of samples (e.g. $N = 10, 100, 1000, 10000$) and compare the results.
test_values = [10**(i+1) for i in range(6)] #a loop based on the power of 10
for i in range(len(test_values)):
  data = priorSampling(test_values[i]) # generate the samples
  P_query = data.count(['f', 'f', 't', 't', 't']) / len(data) # we have to compute P(j,m,a,¬b,¬e), pay attention to the order of the variables
  print("P(j,m,a,¬b,¬e) = ", P_query)

Number or randomly generated events =  10
P(j,m,a,¬b,¬e) =  0.0
Number or randomly generated events =  100
P(j,m,a,¬b,¬e) =  0.0
Number or randomly generated events =  1000
P(j,m,a,¬b,¬e) =  0.003
Number or randomly generated events =  10000
P(j,m,a,¬b,¬e) =  0.0006
Number or randomly generated events =  100000
P(j,m,a,¬b,¬e) =  0.00059
Number or randomly generated events =  1000000
P(j,m,a,¬b,¬e) =  0.00063


---

## EXERCISE2: Pomegranate for Day 2

Use <tt>pomegranate</tt> to calculate the filtered probability of rain on Day 2 when we see an umbrella on Day 1 and Day 2.

What is the filtered probability of rain on Day 2 when we see <tt>not umbrella</tt> on Day 1?

How about if we just have no information about Umbrellas on Day 1 (i.e., only that rain on Day 2)?

In [1]:
!pip install pomegranate==0.15.0

   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ----- ---------------------------------- 2.4/15.8 MB 12.2 MB/s eta 0:00:02
   ----------------------------------- ---- 14.2/15.8 MB 35.5 MB/s eta 0:00:01
   ---------------------------------------- 15.8/15.8 MB 34.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 2.21.0 requires dill<0.3.9,>=0.3.0, but you have dill 0.4.0 which is incompatible.
huggingsound 0.1.6 requires torch!=1.12.0,<1.13.0,>=1.7, but you have torch 2.5.1+cu124 which is incompatible.
neuralset 0.0.1 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
onnx 1.17.0 requires protobuf>=3.20.2, but you have protobuf 3.19.6 which is incompatible.
pyriemann 0.8 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
pysilero-vad 2.1.0 requires onnxruntime<2,>=1.18.0, but you have onnxruntime 1.15.0 which is incompatible.
silero-vad 5.1.2 requires onnxruntime>=1.16.1, but you have onnxruntime 1.15.0 which is incompatible.
tensorboardx 2.6.4 requires protobuf>=3.20, but you have protobuf 3.19.6 w

In [ ]:
from pomegranate import DiscreteDistribution, ConditionalProbabilityTable, Node, BayesianNetwork
# Variables are RainN and UmbrellaN+1 for N = 0, 1, ...
# We have a prior for Rain0, two values 'y'es and 'n'o:
Rain0   = DiscreteDistribution({'y': 0.5, 'n': 0.5})

# Transition model
#
# Conditional distribution relating RainN and RainN+1. Notation for
# the conditional probability table is:
#
# [ 'RainN', 'RainN+1', <probability>]
#
# for the conditional value P(Sprinkler|Cloudy). Note that we have to
# repeat the transition model for each pair of states

Rain1 = ConditionalProbabilityTable(
        [['y', 'y', 0.7],
         ['y', 'n', 0.3],
         ['n', 'y', 0.3],
         ['n', 'n', 0.7]], [Rain0])

Rain2 = ConditionalProbabilityTable(
        [['y', 'y', 0.7],
         ['y', 'n', 0.3],
         ['n', 'y', 0.3],
         ['n', 'n', 0.7]], [Rain1])

# Transition model
#
# Conditional distribution relating RainN and RainN+1. Notation for
# the conditional probability table is:
#
# [ 'RainN', 'RainN+1', <probability>]
#
# for the conditional value P(Sprinkler|Cloudy). Note that we have to
# repeat the transition model for each pair of states
Umbrella1 = ConditionalProbabilityTable(
        [['y', 'y', 0.9],
         ['y', 'n', 0.1],
         ['n', 'y', 0.2],
         ['n', 'n', 0.8]], [Rain1])

Umbrella2 = ConditionalProbabilityTable(
        [['y', 'y', 0.9],
         ['y', 'n', 0.1],
         ['n', 'y', 0.2],
         ['n', 'n', 0.8]], [Rain2])

# The whole network has five nodes:
s1 = Node(Rain0, name="Rain0")
s2 = Node(Rain1, name="Rain1")
s3 = Node(Umbrella1, name="Umbrella1")
s4 = Node(Rain2, name="Rain2")
s5 = Node(Umbrella2, name="Umbrella2")
# Create a network that includes nodes and edges between them:
model = BayesianNetwork("Umbrella Network")
model.add_states(s1, s2, s3, s4, s5)
model.add_edge(s1, s2)
model.add_edge(s2, s3)
model.add_edge(s2, s4)
model.add_edge(s4, s5)
# Fix the model structure
model.bake()

Q2.1 Use <tt>pomegranate</tt> to calculate the filtered probability of rain on Day 2 when we see an umbrella on Day 1 and Day 2.

In [5]:
scenario = [[None,None,'y',None,'y']]
results =  model.predict_proba(scenario)
print(results[0][3].items())

(('n', 0.11664295874822214), ('y', 0.8833570412517778))


Q2.2 What is the filtered probability of rain on Day 2 when we see <tt>not umbrella</tt> on Day 1? (But we still see it in Day 2)

In [6]:
scenario = [[None,None,'n',None,'y']]
results =  model.predict_proba(scenario)
print(results[0][3].items())

(('n', 0.29722921914357675), ('y', 0.7027707808564232))


Q2.3 How about if we just have no information about Umbrellas on Day 1 (But we still see the umbrella in Day 2)?

In [7]:
scenario = [[None,None,None,None,'y']]
results =  model.predict_proba(scenario)
print(results[0][3].items())

(('n', 0.18181818181818196), ('y', 0.8181818181818181))
